# TrOCR Engine (Phase 7)

This notebook is the source of truth for the TrOCR engine. The Python service
in `app/services/trocr/service.py` executes this notebook via `nbclient`.

In [ ]:
import time
from PIL import Image
import numpy as np

MODEL_ID = (parameters or {}).get("model_id", "microsoft/trocr-base-handwritten")
print(f"[trocr_engine] model_id={MODEL_ID}")

In [ ]:
def build_trocr_engine(model_id=MODEL_ID):
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    processor = TrOCRProcessor.from_pretrained(model_id)
    model = VisionEncoderDecoderModel.from_pretrained(model_id)
    return processor, model

def recognize(image, *, engine=None):
    if engine is None:
        engine = build_trocr_engine(model_id=MODEL_ID)
    processor, model = engine
    
    start = time.perf_counter()
    if isinstance(image, np.ndarray):
        import cv2
        if image.ndim == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
    elif isinstance(image, (str, bytes)):
        import io
        image = Image.open(io.BytesIO(image) if isinstance(image, bytes) else image).convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    elapsed = (time.perf_counter() - start) * 1000.0

    w, h = image.size
    box = [[0.0, 0.0], [float(w), 0.0], [float(w), float(h)], [0.0, float(h)]]
    
    return {"boxes": [[box, text, 1.0]], "elapsed_ms": elapsed, "engine": "trocr"}

engine = build_trocr_engine()
print(f"[trocr_engine] ready")